# Teacher's Notebook — Prosthetics Data Engineering Workshop

Run top to bottom before the session. Each markdown cell has the technical context and talking points for the code immediately below it — read before running, not after.

By the end, participants have a working ETL pipeline: messy patient data → cleaned → names SHA-256 hashed → joined to device inventory.

---
## Step 0 — Setup

Make sure `prosthetics_data.csv` is visible in the participant's folder — it's pre-generated, nothing to run.

`pathlib.Path` candidate list rather than a hardcoded string: this solution notebook sits one folder deeper than the participant workbook, so the CSV path differs. The candidate list handles both without any code changes.

In [ ]:
from pathlib import Path
import re
import hashlib
import pandas as pd

: 

In [ ]:
# Find the CSV regardless of which folder this notebook is opened from
csv_candidates = [
    Path('prosthetics_data.csv'),
    Path('..') / 'prosthetics_data.csv',
]
for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Could not find prosthetics_data.csv in this folder or the parent.')

df = pd.read_csv(csv_path)
df.head()

Three calls before any cleaning:
- `df.info()` — column names, dtypes, non-null counts. Messy types show up immediately (object where you'd expect int or datetime).
- `df.isna().sum()` — two columns have NaN: `Full_Name` (PT-007) and `Age_Input` (PT-010).
- `df.head()` — enough rows to see the format variety without overwhelming the screen.

Flags to point out:
- `Age_Input` is dtype `object` despite mostly looking numeric — red flag.
- `Implant_Date` also `object` — free-text dates, six or seven formats incoming.
- `Amputation_Pos` column (position codes: AK, BK, TT, etc.) — acknowledge it, tell them it's out of scope today.

In [ ]:
df.info()
print('\nMissing values per column:')
print(df.isna().sum())

---
## Section 1 — Ingestion & Pipelines (ETL vs ELT)

Frame why pipeline design matters in healthcare before touching any data.

**ETL (Extract, Transform, Load):**
- Validate and mask PII *before* it reaches the destination.
- Right choice when the destination is a shared cloud system and raw records must never land there.
- Trade-off: you need to know your transformations upfront.

**ELT (Extract, Load, Transform):**
- Raw data lands first; transform inside the warehouse later.
- Right choice when data is large and storage is cheap.
- Risk in healthcare: raw patient records sit in the warehouse until the transform runs.

Ask the room: should this charity ETL or ELT when sending records to a cloud analytics platform? Listening for: ETL, because names and PII should be masked *before* hitting the cloud. That's the path this workshop follows.

No code here — conceptual framing only. Move on once a few people have answered.

---
## Section 2 — Data Cleaning

Three cleaning problems, one function each (keeps each piece independently testable):
1. `Age_Input` — integers, birth years, and freeform text all in one column
2. `Implant_Date` — six or seven different date formats including named months with ordinal suffixes
3. `Amputation_Cause` — inconsistent casing and trailing spaces

### 2a — Parse `Age_Input`

Show the raw column first.

In [ ]:
# What does the column actually contain?
df[['Patient_ID', 'Age_Input']]

Four kinds of value in the column:
- Plain age: `28`, `34`, `19`
- Birth year: `1995`, `1988`, `2001` — needs converting to age
- Free text with a number: `'around 50'`, `'maybe 27'`, `"Don't know, maybe 40"`
- `NaN` (PT-010 is genuinely missing)

The logic: NaN → None; pure number > 1900 → birth year, subtract from 2026; ≤ 1900 → already an age; otherwise pull the first digit run from the text.

Worth asking the room what threshold separates age from birth year. 1900 works: no living prosthetics patient was born before it, and no realistic age reaches it.

In [ ]:
CURRENT_YEAR = 2026

def parse_age_input(value):
    """Convert the messy Age_Input column to a clean integer age."""
    if pd.isna(value):
        return None

    # Numeric value — already an int/float, or a string like '34'
    if isinstance(value, (int, float)) and not pd.isna(value):
        numeric_value = int(value)
    else:
        match = re.search(r'\d+', str(value))
        if not match:
            return None
        numeric_value = int(match.group())

    if numeric_value < 100:
        return numeric_value           # already an age
    if numeric_value > 1900:
        return CURRENT_YEAR - numeric_value   # birth year
    return numeric_value               # edge case: 100–1900, return as-is

Test representative inputs before applying to the whole column — catches bugs early and makes the logic visible to the room.

In [ ]:
# Quick sanity checks before applying to the whole column
test_cases = [
    (28,                    28),   # plain integer age
    ('34',                  34),   # age as string
    (1995,                  31),   # birth year
    ("Don't know, maybe 40", 40),  # free text
    ('around 50',           50),
    (float('nan'),         None),  # missing
]
for value, expected in test_cases:
    result = parse_age_input(value)
    status = 'OK' if result == expected else f'FAIL (expected {expected})'
    print(f'{str(value):30} -> {str(result):6}  {status}')

In [ ]:
df['Age_Years'] = df['Age_Input'].apply(parse_age_input)
df[['Patient_ID', 'Age_Input', 'Age_Years']]

### 2b — Parse `Implant_Date`

Show the raw column and count the formats.

In [ ]:
# How many different formats do we actually have?
df['Implant_Date'].unique()

The tempting shortcut: `pd.to_datetime(dayfirst=True)` over everything. The bug: `dayfirst=True` with `format='mixed'` also reorders month/day inside year-first strings — `2023-09-01` becomes January 9th, not September 1st. Demo this live.

In [ ]:
# DEMONSTRATION OF THE PITFALL — do not use this in the actual pipeline
demo = pd.Series(['2023-09-01', '2023/09/01'])
print('With dayfirst=True (WRONG for year-first dates):')
print(pd.to_datetime(demo, format='mixed', dayfirst=True).tolist())
print()
print('With dayfirst=False (correct for year-first, wrong for DD/MM/YYYY):')
print(pd.to_datetime(demo, format='mixed', dayfirst=False).tolist())

No single `pd.to_datetime` call handles both `DD/MM/YYYY` and `YYYY/MM/DD` safely. The fix: detect which category each value is before handing it to pandas.

Two steps:
1. Strip ordinal suffixes (`'5th'` → `'5'`) so `'April 5th 2024'` can parse.
2. Regex-detect year-first format (`^\d{4}[-/]`). If matched, reorder to ISO directly — pandas never gets the chance to misinterpret it.

In [ ]:
# Step 1: strip ordinal suffixes
# 'May 12th 2024' -> 'May 12 2024', 'April 5th 2024' -> 'April 5 2024'
demo_dates = pd.Series(['May 12th 2024', 'April 5th 2024', '2024-06-15'])
demo_dates.str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True)

In [ ]:
# Step 2: the regex that detects year-first formats
# Matches: 2023-09-01, 2024/04/18, 2024-06-15 — anything starting with 4 digits
pattern = r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$'
for v in ['2023-09-01', '2023/09/01', '2024/04/18', '12/05/2024', 'May 12 2024']:
    matched = bool(re.match(pattern, v.strip()))
    print(f'{v:20} year-first? {matched}')

In [ ]:
def parse_implant_date(raw):
    """Return YYYY-MM-DD string for any date format found in this dataset."""
    if pd.isna(raw):
        return None
    s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', str(raw).strip())
    # Year-first: handle before dayfirst=True to avoid month/day swap
    m = re.match(r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$', s)
    if m:
        y, mo, d = m.groups()
        return f'{y}-{int(mo):02d}-{int(d):02d}'
    # Named months and DD/MM/YYYY — pandas can handle these with dayfirst=True
    try:
        return pd.to_datetime(s, dayfirst=True).strftime('%Y-%m-%d')
    except Exception:
        return None

In [ ]:
# Test against every format in the dataset
date_tests = [
    ('12/05/2024',   '2024-05-12'),
    ('2024-06-15',   '2024-06-15'),
    ('May 12th 2024','2024-05-12'),
    ('07/11/2023',   '2023-11-07'),
    ('2023/09/01',   '2023-09-01'),   # the tricky one
    ('March 3 2024', '2024-03-03'),
    ('20/02/2024',   '2024-02-20'),
    ('April 5th 2024','2024-04-05'),
    ('2024/04/18',   '2024-04-18'),
    ('15/07/2024',   '2024-07-15'),
]
for raw, expected in date_tests:
    got = parse_implant_date(raw)
    status = 'OK' if got == expected else f'FAIL (expected {expected})'
    print(f'{raw:22} -> {got}  {status}')

In [ ]:
df['Implant_Date_Clean'] = df['Implant_Date'].apply(parse_implant_date)
df[['Patient_ID', 'Implant_Date', 'Implant_Date_Clean']]

### 2c — Standardise `Amputation_Cause`

Values are correct — just inconsistent casing and whitespace. A single chained call handles it.

- `str.strip()`: removes leading/trailing whitespace
- `str.lower()`: normalises `'TRAUMA'`, `'Trauma'`, `' TRAUMA'` all to `'trauma'`

PT-012 has `'diabets'` — a genuine typo. Worth pointing it out: lowercase/strip doesn't catch spelling errors. That needs a controlled vocabulary or fuzzy matching step, which is out of scope today.

In [ ]:
df['Amputation_Cause_Clean'] = df['Amputation_Cause'].str.lower().str.strip()
df[['Patient_ID', 'Amputation_Cause', 'Amputation_Cause_Clean']]

Run `value_counts()` — `'diabets'` appears alongside `'diabetes'`. Shows the limit of a pure formatting fix.

In [ ]:
df['Amputation_Cause_Clean'].value_counts()

---
## Section 3 — Cybersecurity & Cloud Privacy (PII Masking)

The data is clean but still contains patient names. Before it moves anywhere outside the secure environment, names must go.

Two options: delete the column entirely, or pseudonymise — replace each name with a stable token derived from it. Pseudonymisation lets two datasets still match on the same person without re-exposing the name.

SHA-256: a cryptographic hash function. Same input always produces the same 64-character hex string. One-way — you can't reverse it to recover the original name.

In [ ]:
# DEMO: what SHA-256 looks like
import hashlib

sample = 'John Doe'
digest = hashlib.sha256(sample.strip().lower().encode('utf-8')).hexdigest()
print(f'Input : {sample!r}')
print(f'Hash  : {digest}')
print(f'Length: {len(digest)} characters')

Worth asking why `.strip().lower()` before hashing. `'John Doe'` and `' john doe '` produce different hashes — same person with slightly different casing across two datasets won't match. Normalise first to ensure they do.

Then: why `.encode('utf-8')`? SHA-256 operates on bytes, not strings.

In [ ]:
def anonymize_name(value):
    """Replace a patient name with its SHA-256 hash (normalised first)."""
    normalized = str(value).strip().lower()
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

In [ ]:
# Apply before the join — this is the ETL 'Transform' step protecting PII
df['Full_Name'] = df['Full_Name'].apply(anonymize_name)
df[['Patient_ID', 'Full_Name']].head()

PT-007 had `NaN` in `Full_Name`. After `str(value)` it becomes the string `'nan'`, which hashes consistently. A production system might return a sentinel (`None`) instead — fine to mention as an edge case, but don't let it derail the session. The important concept is the hashing.

---
## Section 4 — Relational Storage & Joins

Clean, privacy-safe records → join to the device inventory to enrich them.

Classic relational structure:
- `Device_Model_ID` is the **foreign key** in the patient table
- `Device_Model_ID` is the **primary key** in the inventory table
- Inner join: keep only rows that match on both sides

Worth asking why inner join over left join. Listening for: inner join drops patients with an unknown device ID, which surfaces data entry errors rather than silently producing NaN columns.

In [ ]:
# Build the reference table
device_inventory = {
    'M210': {'Model_Name': 'AeroStep Lite',   'Cost_USD': 950,  'Manufacturer': 'Stride MedTech'},
    'M302': {'Model_Name': 'FlexFoot Alpha',  'Cost_USD': 1200, 'Manufacturer': 'Global Limb Works'},
    'M404': {'Model_Name': 'CrestFit Pro',    'Cost_USD': 1850, 'Manufacturer': 'OrthoSphere'},
    'M505': {'Model_Name': 'AdaptMax Field',  'Cost_USD': 2100, 'Manufacturer': 'Humanity Mobility Systems'},
}

device_inventory_df = pd.DataFrame.from_dict(device_inventory, orient='index').reset_index()
device_inventory_df = device_inventory_df.rename(columns={'index': 'Device_Model_ID'})
device_inventory_df = device_inventory_df[['Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]
device_inventory_df

A Python dict can't go directly into `pd.merge`. `DataFrame.from_dict(..., orient='index')` converts it — each dict key becomes a row. `reset_index()` makes the key a regular column, then rename it to `Device_Model_ID` so the join has a matching column name on both sides.

In [ ]:
patient_device_df = pd.merge(df, device_inventory_df, on='Device_Model_ID', how='inner')
patient_device_df[['Patient_ID', 'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

In [ ]:
# Confirm no rows were dropped (all 15 patients have a valid device ID)
print(f'Rows before merge: {len(df)}')
print(f'Rows after merge : {len(patient_device_df)}')
assert len(patient_device_df) == 15, 'Row count mismatch — check for unknown Device_Model_IDs'

---
## Wrap-Up

Show the final DataFrame: names replaced with hashes, dates standardised, ages numeric, device details joined in. A complete ETL pipeline in ~50 lines.

Debrief questions:

1. **Anonymisation vs pseudonymisation** — anonymisation means the original can't be recovered. Pseudonymisation means it *can* with the right key. SHA-256 without a salt leans toward anonymisation, but an attacker who knows the likely names can pre-hash them and compare.

2. **Which step was ETL vs ELT?** — all ETL: we transformed (cleaned and masked) before loading.

3. **What cleaning problem did we not solve?** — the `'diabets'` typo. That needs a controlled vocabulary or fuzzy matching.

4. **What breaks if someone adds device `M606`?** — the inner join silently drops that patient. A left join would surface it as NaN instead. Neither is wrong — depends on your quality gate.

In [ ]:
# Final enriched, privacy-safe dataset
output_cols = [
    'Patient_ID', 'Full_Name', 'Age_Years',
    'Implant_Date_Clean', 'Amputation_Cause_Clean',
    'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer',
]
patient_device_df[output_cols]